# scGPT Fine-Tuning Script: PDAC Cell Type Annotation
Fine-tunes scGPT-human on `PDAC_ADJ_Final_Annotated.h5ad` using `scGPT_target_label`.

In [ ]:
# Verify GPU is available
import torch
assert torch.cuda.is_available(), "CUDA is not available"
print("Device:", torch.cuda.get_device_name(0))

Device: NVIDIA A100-SXM4-80GB


In [ ]:
%%capture
# Install dependencies

!pip uninstall -y torch torchtext torchvision torchaudio numpy scanpy

!pip install "numpy<2" -q
!pip install torch==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu121 -q
!pip install --no-deps git+https://github.com/bowang-lab/scGPT.git -q
!pip install "scanpy==1.10.4" wandb -q
!pip install "numpy<2" --force-reinstall --no-deps -q
!pip install --force-reinstall --no-deps torchtext==0.17.2 -q
!pip install scikit-misc -q

print("Restart session")

In [ ]:
# Verify installs after restart
import torch
import numpy as np
import scgpt
import scanpy as sc

print(torch.cuda.is_available())
print(torch.__version__)
print(torch.cuda.get_device_name(0))
print(np.__version__)
print(scgpt.__version__)
print(sc.__version__)

/usr/local/lib/python3.12/dist-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/usr/local/lib/python3.12/dist-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/usr/local/lib/python3.12/dist-packages/scanpy/_utils/__init__.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/usr/local/lib/python3.12/dist-packages/scanpy/__init__.py:36: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/usr/local/lib/python3.12/dist-packages/scanpy/readwrite.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


True
2.2.2+cu121
NVIDIA A100-SXM4-80GB
1.26.4
0.2.4
1.10.4


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify the annotated dataset is accessible
import os
DATA_PATH = "/content/drive/MyDrive/PDAC_ADJ_Final_Annotated.h5ad"
MODEL_DIR = "/content/drive/MyDrive/scGPT_human"

assert os.path.exists(DATA_PATH), f"Data not found at {DATA_PATH}"
assert os.path.exists(MODEL_DIR), f"Model not found at {MODEL_DIR}"
print("Data and model found ✓")

Mounted at /content/drive
Data and model found ✓


In [ ]:
%cd /content

import os
import shutil

if os.path.exists("/content/scGPT_fineTune_protocol"):
    print("Repo already exists, skipping download ")
else:
    !wget -q https://github.com/RCHENLAB/scGPT_fineTune_protocol/archive/refs/heads/main.zip -O /content/protocol.zip
    !unzip -q /content/protocol.zip -d /content/
    !mv /content/scGPT_fineTune_protocol-main /content/scGPT_fineTune_protocol
    !rm /content/protocol.zip
    print("Repo downloaded")

# Copy pre-trained model weights
if not os.path.exists("/content/scGPT_fineTune_protocol/scGPT_human"):
    !cp -r {MODEL_DIR} /content/scGPT_fineTune_protocol/scGPT_human
    print("Model weights copied")
else:
    print("Model weights already present")

# Create datasets folder
os.makedirs("/content/scGPT_fineTune_protocol/datasets", exist_ok=True)

print("Setup complete")

/content
Repo downloaded
Model weights copied
Setup complete


In [ ]:
# Copy file from Drive to local Colab storage first
LOCAL_DATA = "/content/PDAC_ADJ_Final_Annotated.h5ad"

if not os.path.exists(LOCAL_DATA):
    print("Copying data from Drive to local storage...")
    !cp {DATA_PATH} {LOCAL_DATA}
    print("Copy complete ✓")
else:
    print("Local copy already exists ✓")

print("Loading annotated data...")
adata = sc.read_h5ad(LOCAL_DATA)

print(f"Loaded: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")
print(f"Cell type column: {adata.obs['scGPT_target_label'].value_counts().to_dict()}")

# ADJ4, ADJ5, PDAC1 held out as test — matches Sid's split exactly
TEST_SAMPLES = ['ADJ4', 'ADJ5', 'PDAC1']
TRAIN_SAMPLES = [s for s in adata.obs['sample_id'].unique() if s not in TEST_SAMPLES]

print(f"\nTrain samples ({len(TRAIN_SAMPLES)}): {sorted(TRAIN_SAMPLES)}")
print(f"Test  samples ({len(TEST_SAMPLES)}):  {sorted(TEST_SAMPLES)}")

adata_train = adata[adata.obs['sample_id'].isin(TRAIN_SAMPLES)].copy()
adata_test  = adata[adata.obs['sample_id'].isin(TEST_SAMPLES)].copy()

print(f"\nTrain: {adata_train.shape[0]:,} cells")
print(f"Test:  {adata_test.shape[0]:,} cells")

TRAIN_H5AD = "/content/scGPT_fineTune_protocol/datasets/pdac_train.h5ad"
TEST_H5AD  = "/content/scGPT_fineTune_protocol/datasets/pdac_test.h5ad"

adata_train.write_h5ad(TRAIN_H5AD)
adata_test.write_h5ad(TEST_H5AD)

print(f"\nSaved train to {TRAIN_H5AD}")
print(f"Saved test to {TEST_H5AD}")

Copying data from Drive to local storage...
Copy complete ✓
Loading annotated data...
Loaded: 71,074 cells × 36,601 genes
Cell type column: {'T-Cells': 21108, 'Ductal': 11118, 'Fibroblasts': 10050, 'Acinar': 8315, 'Macrophages': 6194, 'B-Cells': 4411, 'Endothelial': 3247, 'Neutrophils': 2368, 'Stellate': 1567, 'Mast': 989, 'Plasma': 708, 'Endocrine': 407, 'NK': 405, 'Schwann': 187, 'Unknown': 0}

Train samples (9): ['ADJ1', 'ADJ2', 'ADJ3', 'ADJ6', 'PDAC2', 'PDAC3', 'PDAC4', 'PDAC5', 'PDAC6']
Test  samples (3):  ['ADJ4', 'ADJ5', 'PDAC1']

Train: 52,481 cells
Test:  18,593 cells

Saved train to /content/scGPT_fineTune_protocol/datasets/pdac_train.h5ad
Saved test to /content/scGPT_fineTune_protocol/datasets/pdac_test.h5ad


In [ ]:
# Preprocess training data for scGPT finetuning
%cd /content/scGPT_fineTune_protocol

!python protocol_preprocess.py \
  --dataset_directory=./datasets/pdac_train.h5ad \
  --cell_type_col="scGPT_target_label" \
  --batch_id_col="sample_id" \
  --load_model=./scGPT_human \
  --wandb_sync=False \
  --wandb_project=pdac_finetune

/content/scGPT_fineTune_protocol
/usr/local/lib/python3.12/dist-packages/scanpy/_utils/__init__.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/usr/local/lib/python3.12/dist-packages/scanpy/__init__.py:36: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/usr/local/lib/python3.12/dist-packages/scanpy/readwrite.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/usr/local/lib/python3.12/dist-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/usr/local/lib/python3.12/dist-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not inst

In [ ]:
# Run finetuning
%cd /content/scGPT_fineTune_protocol

!python protocol_finetune.py \
  --max_seq_len=3001 \
  --include_zero_gene=False \
  --epochs=10   \
  --batch_size=32 \
  --amp=True

/content/scGPT_fineTune_protocol
/usr/local/lib/python3.12/dist-packages/scanpy/_utils/__init__.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/usr/local/lib/python3.12/dist-packages/scanpy/__init__.py:36: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/usr/local/lib/python3.12/dist-packages/scanpy/readwrite.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/usr/local/lib/python3.12/dist-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/usr/local/lib/python3.12/dist-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not inst

In [ ]:
# Save finetuned model to Drive
import glob

# Find the latest training output
save_dirs = sorted(glob.glob("/content/scGPT_fineTune_protocol/save/dev_*"), key=os.path.getmtime)
if not save_dirs:
    raise FileNotFoundError("No training output found")

latest_save = save_dirs[-1]
run_name = os.path.basename(latest_save)  # e.g. "dev_eyeGPT-Apr20-18-31-31"
print(f"Found training output: {latest_save}")
print(f"Contents: {os.listdir(latest_save)}")

# Confirm best_model.pt exists
best_model_path = os.path.join(latest_save, "best_model.pt")
assert os.path.exists(best_model_path), f"best_model.pt not found in {latest_save}"
print(f"best_model.pt size: {os.path.getsize(best_model_path) / 1e6:.1f} MB")

# Save to Drive with run-specific subfolder
DRIVE_SAVE = f"/content/drive/MyDrive/scGPT_human/finetuned_pdac/{run_name}"
os.makedirs(DRIVE_SAVE, exist_ok=True)
shutil.copytree(latest_save, DRIVE_SAVE, dirs_exist_ok=True)
print(f"\nSaved to Drive: {DRIVE_SAVE}")
print(f"Files: {os.listdir(DRIVE_SAVE)}")

Found training output: /content/scGPT_fineTune_protocol/save/dev_eyeGPT-Apr20-18-31-31
Contents: ['vocab.json', 'model_e6.pt', 'id2type.json', 'best_model.pt', 'model_e5.pt', 'model_e1.pt', 'model_e3.pt', 'model_e7.pt', 'dev_train_args.yml', 'protocol_finetune.py', 'run.log']
best_model.pt size: 205.4 MB

Saved to Drive: /content/drive/MyDrive/scGPT_human/finetuned_pdac/dev_eyeGPT-Apr20-18-31-31
Files: ['vocab.json', 'model_e6.pt', 'id2type.json', 'best_model.pt', 'model_e5.pt', 'model_e1.pt', 'model_e3.pt', 'model_e7.pt', 'dev_train_args.yml', 'protocol_finetune.py', 'run.log']


In [ ]:
%cd /content/scGPT_fineTune_protocol

!python protocol_preprocess.py \
  --dataset_directory=./datasets/pdac_test.h5ad \
  --cell_type_col="scGPT_target_label" \
  --batch_id_col="sample_id" \
  --load_model=./scGPT_human \
  --wandb_sync=False \
  --wandb_project=pdac_inference

/content/scGPT_fineTune_protocol
/usr/local/lib/python3.12/dist-packages/scanpy/_utils/__init__.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/usr/local/lib/python3.12/dist-packages/scanpy/__init__.py:36: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/usr/local/lib/python3.12/dist-packages/scanpy/readwrite.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/usr/local/lib/python3.12/dist-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/usr/local/lib/python3.12/dist-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not inst

In [ ]:
from pathlib import Path

# Set working directory
os.chdir("/content/scGPT_fineTune_protocol")

# Find latest saved model directory
save_dirs = sorted(
    glob.glob("/content/scGPT_fineTune_protocol/save/dev_*"),
    key=os.path.getmtime
)

if not save_dirs:
    raise FileNotFoundError("No saved model directories found.")

latest_save = save_dirs[-1]
print(f"Running inference with model: {latest_save}")

# Run inference
!python protocol_inference.py \
    --load_model="{latest_save}" \
    --config="train" \
    --batch_size=32 \
    --wandb_sync=False \
    --wandb_project="pdac_inference"

Running inference with model: /content/scGPT_fineTune_protocol/save/dev_eyeGPT-Apr20-18-31-31
/usr/local/lib/python3.12/dist-packages/scanpy/_utils/__init__.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/usr/local/lib/python3.12/dist-packages/scanpy/__init__.py:36: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/usr/local/lib/python3.12/dist-packages/scanpy/readwrite.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/usr/local/lib/python3.12/dist-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/usr/local/lib/python3.12/dist-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_

In [ ]:
# Find latest inference/eval directories
infer_dirs = sorted(
    glob.glob("/content/scGPT_fineTune_protocol/save/inference_*"),
    key=os.path.getmtime
)
eval_dirs = sorted(
    glob.glob("/content/scGPT_fineTune_protocol/save/eval_*"),
    key=os.path.getmtime
)

results_dirs = infer_dirs if infer_dirs else eval_dirs

if not results_dirs:
    print("Contents of save/:", os.listdir("/content/scGPT_fineTune_protocol/save/"))
else:
    latest_results = results_dirs[-1]
    print(f"Inference output: {latest_results}")
    print("Contents:", os.listdir(latest_results))

    # Copy to Google Drive
    DRIVE_INFER = "/content/drive/MyDrive/scGPT_human/finetuned_pdac/inference"
    os.makedirs(DRIVE_INFER, exist_ok=True)

    shutil.copytree(latest_results, DRIVE_INFER, dirs_exist_ok=True)
    print(f"Saved to Drive: {DRIVE_INFER}")

Inference output: /content/scGPT_fineTune_protocol/save/eval_eyeGPT-Apr20-22-03-47
Contents: ['evaluation_results.json', 'confusion_matrix.png', 'results.pkl', 'predictions.csv', 'umap_predictions.png', 'run.log', 'umap_groundTruth.png']
Saved to Drive: /content/drive/MyDrive/scGPT_human/finetuned_pdac/inference
